```{toctree}
:maxdepth: 2
:caption: Contents:

# Walkthrough Example

This document provides a walkthrough of how to use the `oceanicospy` package to handle observations from various sensors.

```{hint}
If you want to replicate the steps in this walkthrough, you can find the data files in the `data/observations` directory of the repository. On top of that, you need to have the `oceanicospy` package installed in your Python environment along the jupyter notebook extension.

A guide on how to install oceanicospy can be found in the [installation instructions](https://oceanicospy.readthedocs.io/en/latest/installing.html) page of the documentation.

```

# Importing libraries

All the sensors are represented as objects in the module `oceanicospy.observations`.

In [1]:
from oceanicospy.observations import pressure_sensors,weather_stations,AWAC,\
                                    WaveBuoy,ctd,HOBO_Temp,HOBO_TempCond
from datetime import datetime, timedelta
import glob
from pathlib import Path

# Pressure sensors: AQUAlogger, RBR, Bluelog

## Data directories and sampling configuration

For every pressure sensor, we specify the directory where the data files are located and a sampling configuration dictionary that contains parameters such as anchoring depth, sensor height, sampling frequency, burst length, and optional start and end times for filtering the data.

In [ ]:
measurement_pressure_sensors_paths = ['../data/observations/AQUAlogger/',
                                      '../data/observations/RBR/',
                                      '../data/observations/Bluelog/']

sampling_AQ = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=1,burst_length_s=2048,
                    start_time=datetime(2025,5,9,10,0,0),
                    end_time=datetime(2025,5,19,18,0,0))
sampling_RBR = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=7200,
                    start_time=datetime(2025,5,9,10,0,0),
                    end_time=datetime(2025,5,19,18,0,0))
sampling_Bluelog = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=4096,
                        start_time=datetime(2025,7,4,14,0,0), 
                        end_time=datetime(2025,7,10,12,0,0))

## Reading and storing data

In [ ]:
sampling_data = [sampling_AQ,sampling_RBR,sampling_Bluelog]
metadata_list=['AQ','RBR','Bluelog']
dict_raw_measurements = dict()
dict_clean_measurements = dict() 

for idx,measurement_path in enumerate(measurement_pressure_sensors_paths):
    print(measurement_path)
    if 'AQ' in measurement_path:
        object_device = pressure_sensors.AQUAlogger(measurement_path, sampling_data[idx],
                                                    filename='AQUAlogger_520PT5.csv')
    elif 'Bluelog' in measurement_path:
        object_device = pressure_sensors.Bluelog(measurement_path, sampling_data[idx])
    else:
        object_device = pressure_sensors.RBR(measurement_path,sampling_data[idx])
    raw_data = object_device.get_raw_records()
    clean_data = object_device.get_clean_records()
    dict_raw_measurements[metadata_list[idx]] = raw_data
    dict_clean_measurements[metadata_list[idx]] = clean_data

A quick look at the data dictionary can show the available data for each sensor. Each sensor's data is retrieved as a pandas DataFrame, which allows for easy manipulation and analysis. The raw records contain the original columns from the sensor files, which may have different names and formats. The clean records are then obtained by calling `get_clean_records` on each sensor object, which performs several processing steps to prepare the data for analysis. This includes standardizing column names, parsing dates, computing depth from pressure, and assigning burst IDs.

In [ ]:
dict_clean_measurements['AQ'].head()

In [ ]:
dict_clean_measurements['RBR'].head()

In [ ]:
dict_clean_measurements['Bluelog'].head()

The clean records for every sensor derived from the same abstract class share the same column names and formats, which allows for easy comparison and combination of data from different sensors. Note that `depth_aux[m]` may appear alongside `depth[m]` when the sensor provides its own depth estimate and it differs by more than 0.1 m from the hydrostatic computation $d = \frac{(P - P_{atm}) \cdot 10^5}{\rho g}$. Both columns are kept for comparison purposes.

# Weather stations: Davis Vantage Pro, WeatherSens and Rainwise

Currently, there is support for three weather station models: Davis Vantage Pro, WeatherSens, and Rainwise. These stations typically record atmospheric variables such as wind speed and direction, air temperature, relative humidity, barometric pressure, and precipitation.

In [2]:
Davis_weather_station = weather_stations.DavisVantagePro('../data/observations/WeatherStations/DavisVantagePro.txt')
WeatherSens_weather_station = weather_stations.WeatherSens('../data/observations/WeatherStations/WeatherSens.xlsx')
Rainwise_weather_station = weather_stations.Rainwise('../data/observations/WeatherStations/Rainwise.csv')

object_list = [Davis_weather_station,WeatherSens_weather_station,Rainwise_weather_station]

In [3]:
dict_raw_measurements = dict()
dict_clean_measurements = dict()

for ws in object_list:
    raw_data = ws.get_raw_records()
    clean_data = ws.get_clean_records()
    dict_raw_measurements[ws.__class__.__name__] = raw_data
    dict_clean_measurements[ws.__class__.__name__] = clean_data

Calling `get_raw_records()` on each sensor object reads the data files for each weather station and stores the raw records in a dictionary. The raw records contain the original columns from the sensor files, which may have different names and formats. An example of the raw records for the Davis Vantage Pro weather station is shown below.

In [4]:
dict_raw_measurements['DavisVantagePro'].head()

,Date,time,AM/PM,Out,Temp1,Temp2,Hum,Pt.,Speed,Dir1,...,Temp4,Hum2,Dew,Heat,EMC,Density,Samp,Tx,Recept,Int.
0,08/19/23,12:15,AM,---,---,---,---,---,0.0,---,...,28.8,79,24.8,34.2,15.47,1.1319,0,1,0.0,15
1,08/19/23,12:30,AM,---,---,---,---,---,0.0,---,...,28.7,78,24.4,33.7,15.18,1.1329,0,1,0.0,15
2,08/19/23,12:45,AM,---,---,---,---,---,0.0,---,...,28.4,78,24.2,33.1,15.19,1.1339,0,1,0.0,15
3,08/19/23,1:00,AM,---,---,---,---,---,0.0,---,...,28.6,80,24.8,33.8,15.81,1.1321,0,1,0.0,15
4,08/19/23,1:15,AM,---,---,---,---,---,0.0,---,...,28.7,81,25.1,34.3,16.21,1.1307,0,1,0.0,15


A quick look at the column names for each weather station model shows that they are not standardized across models — variable names, units, and datetime formats vary by manufacturer.

In [5]:
for ws in object_list:
    print(ws.__class__.__name__)
    print(dict_raw_measurements[ws.__class__.__name__].info())

DavisVantagePro
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 31 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Date     1218 non-null   object
 1   time     1218 non-null   object
 2   AM/PM    1218 non-null   object
 3   Out      1218 non-null   object
 4   Temp1    1218 non-null   object
 5   Temp2    1218 non-null   object
 6   Hum      1218 non-null   object
 7   Pt.      1218 non-null   object
 8   Speed    1218 non-null   object
 9   Dir1     1218 non-null   object
 10  Run      1218 non-null   object
 11  Speed2   1218 non-null   object
 12  Dir2     1218 non-null   object
 13  Chill    1218 non-null   object
 14  Index1   1218 non-null   object
 15  Index2   1218 non-null   object
 16  Bar      1218 non-null   object
 17  Rain     1218 non-null   object
 18  Rate     1218 non-null   object
 19  D-D1     1218 non-null   object
 20  D-D2     1218 non-null   object
 21  Temp4    1218 non-nul

`get_clean_records()` standardizes column names and dtypes across all models following the `variable[unit]` convention, and applies wind direction conversion where required. Non-recorded values are replaced with `NaN` for consistency.

The clean records for all three stations share a common set of column names, allowing direct comparison across models. Columns exclusive to a particular station (e.g. `solar_radiation[W/m2]` for WeatherSens and Rainwise) are retained when present.

In [6]:
dict_clean_measurements['DavisVantagePro'].head()

,rain[mm],air_temp[C],air_humidity[%],pressure[hPa],wind_speed[m/s],wind_direction[°]
date,,,,,,
2023-08-19 00:15:00,0.0,28.8,NaN,1011.7,0.0,NaN
2023-08-19 00:30:00,0.0,28.7,NaN,1011.6,0.0,NaN
2023-08-19 00:45:00,0.0,28.4,NaN,1011.3,0.0,NaN
2023-08-19 01:00:00,0.0,28.6,NaN,1011.1,0.0,NaN
2023-08-19 01:15:00,0.0,28.7,NaN,1010.8,0.0,NaN


In [7]:
dict_clean_measurements['WeatherSens'].head()

,rain[mm],air_temp[C],air_humidity[%],pressure[hPa],wind_speed[m/s],wind_direction[°],solar_radiation[W/m2]
date,,,,,,,
1999-12-31 19:15:00,0.0,30.500000,82.199997,1010.099976,3.2,80.400002,762.700012
1999-12-31 19:30:00,0.0,30.600000,82.199997,1010.200012,2.6,87.500000,776.000000
1999-12-31 19:45:00,0.0,30.799999,81.800003,1010.299988,2.0,75.400002,812.099976
1999-12-31 20:00:00,0.0,31.200001,80.599998,1010.200012,2.4,73.900002,474.600006
1999-12-31 20:15:00,0.0,31.200001,80.699997,1010.200012,2.1,66.300003,742.599976


In [8]:
dict_clean_measurements['Rainwise'].head()

,rain[mm],air_temp[C],air_humidity[%],pressure[hPa],wind_speed[m/s],wind_direction[°],solar_radiation[W/m2]
date,,,,,,,
2023-11-09 11:59:14,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2023-11-09 12:00:00,0.0,31.4,88.0,1011.3,0.0,337.0,724.0
2023-11-09 12:15:00,0.0,33.9,81.0,1011.2,0.0,337.0,725.0
2023-11-09 12:31:07,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2023-11-09 12:45:00,0.5,34.8,47.0,1010.6,0.0,355.0,4.0


In [9]:
for ws in object_list:
    print(ws.__class__.__name__)
    print(dict_clean_measurements[ws.__class__.__name__].info())

DavisVantagePro
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1218 entries, 2023-08-19 00:15:00 to 2023-08-31 16:30:00
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rain[mm]           1218 non-null   float64
 1   air_temp[C]        1218 non-null   float64
 2   air_humidity[%]    0 non-null      float64
 3   pressure[hPa]      1218 non-null   float64
 4   wind_speed[m/s]    1218 non-null   float64
 5   wind_direction[°]  869 non-null    float64
dtypes: float64(6)
memory usage: 66.6 KB
None
WeatherSens
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 44064 entries, 1999-12-31 19:15:00 to 2025-07-10 16:45:00
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   rain[mm]               44064 non-null  float64
 1   air_temp[C]            44064 non-null  float64
 2   air_humidity[%]        44063 non-null  float

# CTD sensors

Currently, there is support for two CTD sensor models: `SeaSunMarineTechCTD` and `CastawayCTD`. Both record vertical profiles of water column properties as a function of depth, including temperature, conductivity, pressure, and salinity.

In [ ]:
sea_sun_marine_tech = ctd.SeaSunMarineTechCTD('../data/observations/CTD/SeaSunMarineTech.TOB')
castaway = ctd.CastawayCTD('../data/observations/ctd/CC2349006_20240301_185500.csv',has_header=False)

When the file recorded by the CastAway CTD does not contain an embedded header, set `has_header=False` in the constructor. In this case, the class will look for metadata in a summary file generated by the CastAway software, which contains cast date and time, location, and sensor model. If the header is present in the file, set `has_header=True` and the metadata will be read directly from it.

In [ ]:
ctd_objects = [sea_sun_marine_tech, castaway]

In [ ]:
dict_ctd_raw_measurements = dict()
dict_ctd_clean_measurements = dict()

for ctd_obj in ctd_objects:
    print(ctd_obj.cast_time)
    raw_data = ctd_obj.get_raw_records()
    clean_data = ctd_obj.get_clean_records()
    dict_ctd_raw_measurements[ctd_obj.__class__.__name__] = raw_data
    dict_ctd_clean_measurements[ctd_obj.__class__.__name__] = clean_data

`cast_time` is a parsed metadata property available on all CTD objects, representing the UTC start time of the cast.

In [ ]:
dict_ctd_raw_measurements['SeaSunMarineTechCTD'].head()

In [ ]:
dict_ctd_raw_measurements['CastawayCTD'].head()

Similar to the previous cases with other sensors, when we use the raw records, we can see that the column names and formats are different for each CTD sensor model.

In [ ]:
for key in dict_ctd_raw_measurements.keys():
    print(f"Raw records for {key}:")
    print(dict_ctd_raw_measurements[key].columns)
    print("\n")

Because of the different formats and column names in the raw records, it is not straightforward to compare or combine data from different CTD sensors. However, by using the `get_clean_records` method of each CTD sensor class, we can standardize the column names and formats, which allows for easy comparison and combination of data from different CTD sensors.

In [ ]:
dict_ctd_clean_measurements['SeaSunMarineTechCTD'].head()

In [ ]:
dict_ctd_clean_measurements['CastawayCTD'].head()

If the header is present in the file, the class will look for metadata information in the header of the file. This header contains information about the cast, such as the date and time of the cast, the location, etc.

In [ ]:
ctd_castaway_header = ctd.CastawayCTD('../data/observations/ctd/CastAway_with_header.csv',has_header=True)
ctd_castaway_header_raw = ctd_castaway_header.get_raw_records()
ctd_castaway_header_clean = ctd_castaway_header.get_clean_records()

In [ ]:
ctd_castaway_header_clean.head()

# ADCP sensors: AWAC

Support for ADCP sensors, such as the AWAC, is also included in the module. The `AWAC` class is designed to handle the specific data formats and requirements of ADCP measurements, allowing users to easily load and process wave records from AWAC deployments.

Similar to the pressure sensors, it requires a directory path and a sampling configuration dictionary. The directory must contain the `.hdr` and `.wad` files exported by the AWAC instrument.

In [ ]:
measurement_AWAC_paths = ['../data/observations/AWAC/']

sampling_AWAC = dict(
    anchoring_depth=1,
    sensor_height=0.2,
    sampling_freq=1,
    burst_length_s=2048,
    start_time=datetime(2023,8,18,19,0,0),
    end_time=datetime(2023,9,1,15,0,0) - timedelta(seconds=1))

In [ ]:
AWAC_obj = AWAC(measurement_AWAC_paths[0],sampling_AWAC)

## Reading wave records

The method `get_raw_wave_records` can be used to retrieve the raw wave records. The `from_single_wad` parameter controls whether data is read from a single `.wad` file containing all bursts (`True`) or from multiple per-burst `.wad` files (`False`).

### Using the .wad file with all the records

In [ ]:
df_raw_wave_single_wad = AWAC_obj.get_raw_wave_records(from_single_wad=True)
df_clean_wave_single_wad = AWAC_obj.get_clean_wave_records(from_single_wad=True)

In [ ]:
df_raw_wave_single_wad.head()

The method `get_clean_wave_records()` parses the datetime index, assigns burst IDs, and standardizes column names and pressure units.

In [ ]:
df_clean_wave_single_wad.head()

### Using .wad files recorded per burst

In [ ]:
df_raw_wave_combined_wad = AWAC_obj.get_raw_wave_records(from_single_wad=False)
df_clean_wave_combined_wad = AWAC_obj.get_clean_wave_records(from_single_wad=False)

In [ ]:
df_raw_wave_combined_wad.head()

In [ ]:
df_clean_wave_combined_wad.head()

## Reading current records

Current velocity records are stored in `.v1` and `.v2` files, containing the east (x) and north (y) velocity components respectively. Each column represents a depth cell (`cell_1`, `cell_2`, ..., `cell_N`), where N is the number of cells defined in the `.hdr` file.

In [ ]:
x_currents_raw,y_currents_raw = AWAC_obj.get_raw_currents_records()

In [ ]:
x_currents_raw.head()

In [ ]:
y_currents_raw.head()

The method `get_clean_currents_records` allows users to retrieve cleaned current records, which can be useful for further analysis and interpretation of the data. If the `compute_speed_dir` parameter is set to `True`, the method will also compute the current speed and direction based on the u and v components of the currents.

In [ ]:
x_currents_clean,y_currents_clean,current_speed,current_dir = AWAC_obj.get_clean_currents_records()

In [ ]:
x_currents_clean.head()

In [ ]:
current_dir.head()

# Spotter buoy

The `Buoy` class supports Spotter buoy data exported in two formats: Sofar Ocean (epoch-based) and AQUAlink (ISO 8601 timestamp). The source format is detected automatically from the column names in the CSV file.

In [ ]:
csv_files_spotter = glob.glob('../data/observations/Buoy/*.csv')
dict_raw_spotter_sources = dict()
dict_clean_spotter_sources = dict()
for csv_file in csv_files_spotter:
    # Note: format is inferred from the filename. Files containing 'sofar'
    # are treated as Sofar exports; all others as AQUAlink exports.
    if 'sofar' in csv_file:
        sampling_spotter = dict(start_time=datetime(2025,5,11,0,0,0),
                        end_time=datetime(2025,5,21,15,0,0))
    else:
        sampling_spotter = dict(start_time=datetime(2023,5,11,0,0,0),
                                end_time=datetime(2023,5,21,15,0,0))

    source_name = Path(csv_file).stem
    spotter_object = WaveBuoy(csv_file,sampling_spotter)
    dict_raw_spotter_sources[source_name] = spotter_object.get_raw_records()
    dict_clean_spotter_sources[source_name] = spotter_object.get_clean_records()

In [ ]:
dict_raw_spotter_sources['spotter_data_aqualink'].head()

In [ ]:
dict_raw_spotter_sources['spotter_data_sofar'].head()

In [ ]:
for key in dict_raw_spotter_sources.keys():
    print(f"Raw records for {key}:")
    print(dict_raw_spotter_sources[key].info())

The clean records for the Spotter buoy are obtained by applying the `get_clean_records` method of the `Buoy` class, which performs several processing steps to prepare the data for analysis. This includes standardizing column names to follow a consistent naming convention, converting all columns that contain "spotter" in their name to float data type.

In [ ]:
dict_clean_spotter_sources['spotter_data_aqualink'].head()

In [ ]:
dict_clean_spotter_sources['spotter_data_sofar'].head()

The column names can be inspected once the standardization is applied, and they follow a consistent naming convention that includes the variable name and the unit of measurement in square brackets.

In [ ]:
for key in dict_clean_spotter_sources.keys():
    print(f"Clean records for {key}:")
    print(dict_clean_spotter_sources[key].info())

# HOBO sensors

Currently, there is support for two HOBO logger models: `HOBO_Temp` for temperature-only recordings and `HOBO_TempCond` for combined temperature and conductivity measurements.

In [ ]:
obj_HOBO_temp = HOBO_Temp('../data/observations/HOBO/TL.csv',
                          start_dt=datetime(2025,3,16,0,0,0), 
                          end_dt=datetime(2025,3,18,0,0,0))

obj_HOBO_temp_cond = HOBO_TempCond('../data/observations/HOBO/CL.csv',
                            start_dt=datetime(2024,9,13,0,0,0), 
                            end_dt=datetime(2024,9,18,0,0,0))

In [ ]:
dict_hobo_raw_measurements = dict()
dict_hobo_clean_measurements = dict()
for hobo in [obj_HOBO_temp, obj_HOBO_temp_cond]:
    raw_data = hobo.get_raw_records()
    clean_data = hobo.get_clean_records()
    dict_hobo_raw_measurements[hobo.__class__.__name__] = raw_data
    dict_hobo_clean_measurements[hobo.__class__.__name__] = clean_data

In [ ]:
dict_hobo_raw_measurements['HOBO_Temp'].head()

In [ ]:
dict_hobo_raw_measurements['HOBO_TempCond'].head()

`get_clean_records()` standardizes column names, parses the datetime index, and retains only the relevant measurement columns (`temperature[C]`, `conductivity[uS/cm]`, `seq`).

In [ ]:
for key in dict_hobo_raw_measurements.keys():
    print(f"\tRaw records for {key}:")
    print(dict_hobo_raw_measurements[key].info())

In [ ]:
dict_hobo_clean_measurements['HOBO_Temp'].head()

In [ ]:
dict_hobo_clean_measurements['HOBO_TempCond'].head()

 The column names are standardized to follow a consistent naming convention that includes the variable name and the unit of measurement in square brackets. The date column is parsed using a specific format to ensure that it is correctly interpreted as a datetime object. Finally, the date column is set as the index of the DataFrame and sorted in chronological order.

In [ ]:
for key in dict_hobo_clean_measurements.keys():
    print(f"\tClean records for {key}:")
    print(dict_hobo_clean_measurements[key].info())